# FastText Embeddings Training

Train domain-specific FastText embeddings on Steam reviews for sentiment classification.

In [1]:
import pandas as pd
import fasttext
from pathlib import Path
from tqdm import tqdm

# Paths
BASE_PATH = Path('../steam_data_20251208')
REVIEWS_PATH = BASE_PATH / 'reviews'
OUTPUT_PATH = Path('../models')
OUTPUT_PATH.mkdir(exist_ok=True)

## 1. Load All Reviews

In [2]:
# Load all reviews from all genres
all_reviews = []
genre_folders = [f for f in REVIEWS_PATH.iterdir() if f.is_dir()]

for genre_folder in tqdm(genre_folders, desc='Loading reviews'):
    for csv_file in genre_folder.glob('reviews_*.csv'):
        try:
            df = pd.read_csv(csv_file)
            if 'review_text' in df.columns:
                all_reviews.append(df[['review_text']])
        except Exception as e:
            print(f"Error loading {csv_file.name}: {e}")

reviews_df = pd.concat(all_reviews, ignore_index=True)
print(f"Loaded {len(reviews_df):,} reviews")

Loading reviews: 100%|██████████| 12/12 [00:01<00:00,  7.19it/s]

Loaded 103,946 reviews


## 2. Preprocessing

In [3]:
import sys
sys.path.insert(0, '..')
from preprocessing import preprocess_for_fasttext

# Test preprocessing
sample = "This game is TERRIBLE!!! Don't buy it... https://store.steam.com <br/>"
print(f"Before: {sample}")
print(f"After:  {preprocess_for_fasttext(sample)}")

Before: This game is TERRIBLE!!! Don't buy it... https://store.steam.com <br/>
After:  this game is terrible ! ! ! don ' t buy it . . .


In [4]:
# Apply preprocessing
tqdm.pandas(desc='Preprocessing')
reviews_df['processed'] = reviews_df['review_text'].progress_apply(preprocess_for_fasttext)

# Filter empty reviews
reviews_df = reviews_df[reviews_df['processed'].str.len() > 10]
print(f"Reviews after filtering: {len(reviews_df):,}")

Preprocessing: 100%|██████████| 103946/103946 [00:01<00:00, 52740.84it/s]


Reviews after filtering: 93,099


## 3. Prepare Training File

In [5]:
# Save to text file (one review per line - FastText format)
corpus_path = OUTPUT_PATH / 'fasttext_corpus.txt'

with open(corpus_path, 'w', encoding='utf-8') as f:
    for review in tqdm(reviews_df['processed'], desc='Writing corpus'):
        f.write(review + '\n')

print(f"Corpus saved to {corpus_path}")

Writing corpus: 100%|██████████| 93099/93099 [00:00<00:00, 1117585.55it/s]

Corpus saved to ..\models\fasttext_corpus.txt


## 4. Train FastText Model

In [6]:
# Train FastText
model = fasttext.train_unsupervised(
    str(corpus_path),
    model='skipgram',
    dim=200,
    lr=0.05,
    epoch=10,
    minCount=3,
    wordNgrams=2, # Use bigrams
)

print(f"Vocabulary size: {len(model.words):,}")

Vocabulary size: 32,313


## 5. Save Model

In [7]:
# Save model
model_path = OUTPUT_PATH / 'steam_fasttext.bin'
model.save_model(str(model_path))
print(f"Model saved to {model_path}")

Model saved to ..\models\steam_fasttext.bin


## 6. Test Embeddings

In [8]:
# Test word vectors
test_words = ['game', 'fun', 'boring', 'recommend', 'refund', 'dlc', 'grind', 'masterpiece']

print("Most similar words:")
for word in test_words:
    similar = model.get_nearest_neighbors(word, k=5)
    print(f"\n{word}:")
    for score, w in similar:
        print(f"  {w}: {score:.3f}")

Most similar words:

game:
  this: 0.844
  it: 0.797
  but: 0.730
  is: 0.723
  the: 0.711

fun:
  enjoyable: 0.677
  😎: 0.675
  great: 0.653
  fun~: 0.637
  good: 0.636

boring:
  -boring: 0.743
  repetive: 0.728
  uninteresting: 0.702
  repetitive: 0.698
  repetetive: 0.684

recommend:
  recommmend: 0.936
  recommends: 0.861
  reccommend: 0.840
  recommened: 0.810
  recommed: 0.804

refund:
  refunds: 0.807
  refunded: 0.760
  refunding: 0.747
  requested: 0.619
  2hr: 0.591

dlc:
  dlcs: 0.712
  £12: 0.560
  transmission: 0.545
  150$: 0.543
  expansions: 0.532

grind:
  grind-: 0.878
  grind-y: 0.841
  grindin: 0.800
  grind-fest: 0.763
  grinding: 0.751

masterpiece:
  masterpieces: 0.907
  masterpice: 0.896
  centerpiece: 0.789
  masterpeace: 0.753
  masterful: 0.603


In [9]:
# Test sentence embedding
sentence = "this game is absolutely amazing"
embedding = model.get_sentence_vector(sentence)
print(f"Sentence: '{sentence}'")
print(f"Embedding shape: {embedding.shape}")
print(f"Embedding (first 10): {embedding[:10]}")

Sentence: 'this game is absolutely amazing'
Embedding shape: (200,)
Embedding (first 10): [-0.04568204 -0.01047201 -0.03869694 -0.00284534  0.00491289 -0.01686753
  0.06126443 -0.03994106 -0.02235871 -0.00340288]
